# Follow-up questions

Today every question starts from zero. Ask "only rock" on its own and the model has to guess what "only" narrows.

The fix is to put earlier turns in the prompt. This notebook builds that prompt by hand first, then runs it through the real graph.

In [3]:
# Jupyter starts in notebooks/, so `backend` isn't importable until the repo root is on sys.path.
# A script run as `python -m ...` from the root gets this for free.
import sys
from pathlib import Path

root = str(Path.cwd().parent)
if root not in sys.path:
    sys.path.insert(0, root)

from backend.graph.build import graph, initial_state
from backend.graph.nodes import format_history
from backend.graph.state import Turn

# the shape didn't change: history is new input, not a new node
print(graph.get_graph().draw_ascii())

                                +-----------+                                 
                                | __start__ |                                 
                                +-----------+                                 
                                       *                                      
                                       *                                      
                                       *                                      
                                +------------+                                
                                | understand |                                
                                +------------+                                
                                       *                                      
                                       *                                      
                                       *                                      
                             +-----------------+    

## The problem

A follow-up with no history. Expect the model to invent a question around the words.

In [4]:
state = await graph.ainvoke(initial_state("only rock"))
print(state["status"], "|", state["sql"])

ok | SELECT p.name
FROM playlist p
WHERE EXISTS (SELECT 1 FROM playlist_track pt WHERE pt.playlist_id = p.playlist_id)
  AND NOT EXISTS (
        SELECT 1
        FROM playlist_track pt
        JOIN track t ON t.track_id = pt.track_id
        JOIN genre g ON g.genre_id = t.genre_id
        WHERE pt.playlist_id = p.playlist_id
          AND g.name <> 'Rock'
      );


## What goes in the prompt

Each turn gives its question and final SQL. Result rows stay out, because they cost tokens and the data has never gone to the LLM. A failed turn's SQL is dropped too, since it's wrong and the model would copy it.

Only the last 5 turns go in.

In [5]:
history = [
    Turn(question="Top 5 artists by number of tracks",
         sql='SELECT ar.name, COUNT(*) AS n FROM artist ar JOIN album al ON al.artist_id = ar.artist_id JOIN track t ON t.album_id = al.album_id GROUP BY ar.name ORDER BY n DESC LIMIT 5',
         status="ok"),
    Turn(question="Which customers spent the most?", sql="", status="failed"),
]
print(format_history(history))

Earlier in this conversation (oldest first):
Q: Top 5 artists by number of tracks
SQL: SELECT ar.name, COUNT(*) AS n FROM artist ar JOIN album al ON al.artist_id = ar.artist_id JOIN track t ON t.album_id = al.album_id GROUP BY ar.name ORDER BY n DESC LIMIT 5
Q: Which customers spent the most?
SQL: none, this turn failed
The question below may be a follow-up to these turns. If it refers back to them ("only rock", "now compare with 2022"), build on the most recent relevant SQL. If it's a new question, ignore them.




## Through the graph

A small helper runs a question with the history so far and returns the new turn, so each pair reads like a conversation.

In [6]:
async def ask(question, history):
    state = await graph.ainvoke(initial_state(question, history))
    print('Q:', question)
    print('  ', state['status'], '|', state['attempts'], 'repair(s)')
    print('  ', state['sql'])
    print('   rows:', state['result'][:3])
    print()
    return history + [Turn(question=question, sql=state['sql'], status=state['status'])]

In [7]:
chat = await ask("Top 5 artists by number of tracks", [])
chat = await ask("only rock", chat)

Q: Top 5 artists by number of tracks
   ok | 0 repair(s)
   SELECT a.name AS artist_name, COUNT(t.track_id) AS track_count
FROM artist a
LEFT JOIN album al ON a.artist_id = al.artist_id
LEFT JOIN track t ON al.album_id = t.album_id
GROUP BY a.artist_id, a.name
ORDER BY track_count DESC, a.name
LIMIT 5;
   rows: [{'artist_name': 'Iron Maiden', 'track_count': 213}, {'artist_name': 'U2', 'track_count': 135}, {'artist_name': 'Led Zeppelin', 'track_count': 114}]

Q: only rock
   ok | 0 repair(s)
   SELECT a.name AS artist_name, COUNT(t.track_id) AS track_count
FROM artist a
LEFT JOIN album al ON a.artist_id = al.artist_id
LEFT JOIN track t ON al.album_id = t.album_id
LEFT JOIN genre g ON t.genre_id = g.genre_id
WHERE g.name = 'Rock'
GROUP BY a.artist_id, a.name
ORDER BY track_count DESC, a.name
LIMIT 5;
   rows: [{'artist_name': 'Led Zeppelin', 'track_count': 114}, {'artist_name': 'U2', 'track_count': 112}, {'artist_name': 'Deep Purple', 'track_count': 92}]



In [8]:
chat = await ask("How many invoices were there in 2021?", [])
chat = await ask("now compare with 2022", chat)

Q: How many invoices were there in 2021?
   ok | 0 repair(s)
   SELECT COUNT(*) AS invoice_count
FROM invoice
WHERE EXTRACT(YEAR FROM invoice_date) = 2021;
   rows: [{'invoice_count': 83}]

Q: now compare with 2022
   ok | 0 repair(s)
   SELECT EXTRACT(YEAR FROM invoice_date) AS year, COUNT(*) AS invoice_count
FROM invoice
WHERE EXTRACT(YEAR FROM invoice_date) IN (2021, 2022)
GROUP BY EXTRACT(YEAR FROM invoice_date)
ORDER BY year;
   rows: [{'year': Decimal('2021'), 'invoice_count': 83}, {'year': Decimal('2022'), 'invoice_count': 83}]



## A new question in the same chat

History shouldn't leak into an unrelated question. This SQL should ignore the invoice turns above.

In [9]:
chat = await ask("How many employees are there?", chat)

Q: How many employees are there?
   ok | 0 repair(s)
   SELECT COUNT(*) AS employee_count
FROM employee;
   rows: [{'employee_count': 8}]



## Why not a LangGraph checkpointer

A checkpointer saves the whole graph state per thread and reloads it on the next call. We already save every turn in the `messages` table for the chat UI. A checkpointer would store the same conversation a second time, in a second format, and the two could disagree.

So history is plain input: block 5 reads the last turns from `messages` and passes them to `initial_state`.